# ctxblame on Google Colab (Zero-API-Cost via vLLM)

Run `ctxblame` experiments and the **Phase 3 attribution benchmark** (`ctxblame` vs. unconditional `ctxblame` vs. leave-one-out ablation vs. LLM self-explanation) at **$0 API cost** on a Colab **L4 (24 GB)** or **A100 (40 GB)** GPU with **vLLM automatic prefix caching** enabled.

Recommended runtime: **Runtime → Change runtime type → L4 GPU** (or **A100 GPU** with Google AI Pro).

In [ ]:
!nvidia-smi
# Optional: mount Google Drive to persist traces & benchmark reports across Colab sessions
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Install vLLM and clone/install ctxblame
REPO_URL = "https://github.com/kekeandzeyu/ctxblame.git"
!pip install -q vllm
!rm -rf /content/ctxblame && git clone {REPO_URL} /content/ctxblame
!pip install -q -e /content/ctxblame

In [ ]:
# Start vLLM in the background with automatic prefix caching + tool calling.
# Recommended models:
#   - L4 (24 GB):  Qwen/Qwen2.5-Coder-7B-Instruct  (fast) or Qwen/Qwen3-8B
#   - A100 (40GB): Qwen/Qwen3-14B or Qwen/Qwen2.5-Coder-14B-Instruct
import subprocess, time, urllib.request

MODEL = "Qwen/Qwen3-8B"
TOOL_PARSER = "hermes"  # works for Qwen3 & Qwen2.5 tool calling in vLLM

vllm_log = open("/tmp/vllm.log", "w")
vllm_proc = subprocess.Popen([
    "vllm", "serve", MODEL,
    "--host", "127.0.0.1", "--port", "8000",
    "--enable-prefix-caching",
    "--enable-auto-tool-choice",
    "--tool-call-parser", TOOL_PARSER,
    "--max-model-len", "32768",
    "--gpu-memory-utilization", "0.90",
], stdout=vllm_log, stderr=subprocess.STDOUT)

for _ in range(180):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=2) as r:
            if r.status == 200:
                print("vLLM is ready!", r.read().decode()[:200])
                break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("vLLM failed to start; inspect /tmp/vllm.log")

In [ ]:
# 1) Quick sanity check: conditional ctxblame on Phase 1 step5 (precondition + 3-way OR-carriers)
%cd /content/ctxblame
!ctxblame why experiments/phase1/results/traces/openai-1.jsonl \
    --step 5 \
    --key-arg 'command~^python -m pytest' \
    --base-url http://127.0.0.1:8000/v1 \
    --model {MODEL} \
    --max-tokens 2048 \
    --no-thinking-param \
    --free \
    --concurrency 8

In [ ]:
# 2) Run the full Step 4 / Phase 3 Benchmark (ctxblame vs. ctxblame_uncond vs. LOO vs. self_explain)
# Tip: pass `--only p3` first for a fast ~5-minute run on the 5 synthetic stress cases,
# then remove `--only` to run all 12 cases (Phase 0 + Phase 1 + Phase 3).
%cd /content/ctxblame
!python experiments/phase3/benchmark.py \
    --base-url http://127.0.0.1:8000/v1 \
    --model {MODEL} \
    --max-tokens 2048 \
    --no-thinking-param \
    --free \
    --concurrency 8 \
    --only p3

In [ ]:
# Render the generated benchmark table right in Colab
from IPython.display import Markdown, display
display(Markdown(open("/content/ctxblame/experiments/phase3/results/benchmark.md").read()))

In [ ]:
# 3) Optional: record a brand-new multi-turn coding agent trace against Colab vLLM
import subprocess, time

TRACE = "/content/ctxblame/traces/colab-agent.jsonl"
proxy_proc = subprocess.Popen([
    "ctxblame", "record",
    "--upstream", "http://127.0.0.1:8000/v1",
    "--port", "8791",
    "--trace", TRACE,
])
time.sleep(2)

!python experiments/phase1/agent.py \
    --format openai \
    --base-url http://127.0.0.1:8791 \
    --model {MODEL} \
    --no-thinking \
    --max-tokens 2048 \
    --workspace /tmp/ws

proxy_proc.terminate()
!ctxblame steps {TRACE}

In [ ]:
# 4) Inspect vLLM prefix-cache hit metrics directly from the server
!curl -s http://127.0.0.1:8000/metrics | grep -E "prefix_cache|prompt_tokens|generation_tokens" | head -n 20